# Phase 1/2 kickoff — Open-i data exploration

Run this notebook in **Google Colab** (Runtime doesn't need a GPU for this part — CPU is fine).

This pulls the real Open-i radiology reports, runs the repo's `download_openi.py` + `preprocess.py` scripts, and pokes at the resulting structured data so we know what we're actually fine-tuning on before we write any training code.

## 1. Mount Drive and clone the repo there

We're cloning into Google Drive (`/content/drive/MyDrive/Clinical_Report_Assistant`) instead of Colab's local `/content` disk, so the repo, downloaded data, and later fine-tuning checkpoints all survive runtime disconnects instead of vanishing every time the VM resets.

Since the repo is private, Colab also needs a GitHub token to clone it. **Don't paste your token directly into a cell** — use Colab's built-in Secrets manager instead:

1. Click the key icon (🔑) in the left sidebar of Colab.
2. Add a new secret named `GH_TOKEN`, paste your GitHub PAT as the value, and toggle "Notebook access" on.
3. Run the cell below. It'll pop up a Drive authorization prompt the first time — approve it, then it clones (or, on later runs, pulls the latest) into your Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os

from google.colab import userdata

GH_TOKEN = userdata.get("GH_TOKEN")
REPO_DIR = "/content/drive/MyDrive/Clinical_Report_Assistant"
REPO_URL = f"https://{GH_TOKEN}@github.com/ozgurberat/clinical-report-assistant.git"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already exists in Drive — pulling latest instead of re-cloning.")
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL} "{REPO_DIR}"
    %cd {REPO_DIR}

## 2. Install dependencies

Just the bits needed for the data pipeline — not the full fine-tuning stack yet.

In [ ]:
!pip install -q pyyaml requests tqdm lxml pandas

## 3. Download the real dataset

This hits the official NLM Open-i host directly — couldn't be reached from my sandbox earlier, but Colab has normal internet access so this should just work. Since we're inside the Drive-mounted repo folder, the downloaded archives land in Drive too and won't need re-downloading next session.

In [ ]:
!python -m src.data.download_openi --out data/raw

If that cell fails to connect, fall back to the Kaggle mirror instead:
```
!pip install -q kaggle
# upload your kaggle.json (Account -> Create New Token on kaggle.com) via the Colab file browser first, then:
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!python -m src.data.download_openi --out data/raw --source kaggle
```

## 4. Parse the raw XML into a structured corpus

This runs `src/data/preprocess.py`, which we already unit-tested against a synthetic sample — now it runs against the real ~4k reports.

In [ ]:
!python -m src.data.preprocess --raw data/raw/reports --out data/processed

## 5. Load it and look around

In [ ]:
import json
import pandas as pd

records = [json.loads(line) for line in open("data/processed/reports.jsonl")]
df = pd.DataFrame(records)

print(f"Total reports: {len(df)}")
df.head()

### Look at one full report

Worth actually reading a couple of these end to end before writing any training code — get a feel for how radiologists phrase findings vs. impressions.

In [ ]:
sample = records[0]
for key, value in sample.items():
    print(f"--- {key} ---")
    print(value)
    print()

### Field length distribution

Matters for fine-tuning: this tells us how long to set the model's max sequence length, and whether findings/impression are wildly different lengths (they usually are — impression is typically much shorter, a one-line summary of findings).

In [ ]:
df["findings_len"] = df["findings"].str.len()
df["impression_len"] = df["impression"].str.len()
df[["findings_len", "impression_len"]].describe()

### Missing fields

The data card already flagged that COMPARISON/INDICATION are often empty — let's see exactly how often, since it affects how we structure the fine-tuning prompt (e.g. do we always include a COMPARISON field even when it's blank, or drop it).

In [ ]:
for col in ["comparison", "indication", "findings", "impression"]:
    missing = (df[col].str.strip() == "").sum()
    print(f"{col:12s}: {missing:4d} / {len(df)} empty ({missing/len(df):.1%})")

### Most common MeSH diagnosis terms

This is basically the label distribution for the dataset — useful to know if it's dominated by "normal" (very likely) and how long-tailed the actual findings are, since that affects how well fine-tuning will generalize to rarer conditions.

In [ ]:
from collections import Counter

all_mesh = [term for terms in df["mesh_major"] for term in terms]
Counter(all_mesh).most_common(20)

### Duplicate-content artifacts

The `impression_len` outlier we inspected by hand turned out to contain an entire second report (header + findings + impression) embedded inside the impression field — a data export artifact, not a genuinely long clinical impression. `FINDINGS:`, `IMPRESSION:`, and `EXAM:` should never legitimately appear as literal text *inside* the findings/impression fields themselves, so their presence is a strong signal of this duplication bug. This checks how widespread it is before we decide whether it's a one-off to drop or needs a real cleaning rule in `preprocess.py`.

In [ ]:
import re

HEADER_PATTERN = re.compile(r"\b(FINDINGS|IMPRESSION|EXAM)\s*:", flags=re.IGNORECASE)

for col in ["findings", "impression"]:
    flagged = df[df[col].apply(lambda t: bool(HEADER_PATTERN.search(t)))]
    print(f"{col:10s}: {len(flagged):4d} / {len(df)} reports ({len(flagged)/len(df):.1%}) contain an embedded section header")

df[df["impression"].apply(lambda t: bool(HEADER_PATTERN.search(t)))][["report_id", "impression"]].head(10)

## 6. Build and inspect the fine-tuning training data

`preprocess.py` now drops the duplicate-artifact report automatically. This step builds the actual chat-formatted training files for both fine-tuning tasks (extraction, summarization), then checks them two ways before any GPU time gets spent: eyeballing a couple of real examples, and an automated pass over *every* example checking for the specific failure modes we already know to watch for — malformed JSON targets, leaked `XXXX` tokens, empty targets, wrong message role order.

In [ ]:
!python -m src.finetuning.prompt_format --processed data/processed

### Eyeball a couple of real examples

In [ ]:
def show_examples(path, n=2):
    print(f"=== {path} ===")
    with open(path) as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            example = json.loads(line)
            print(f"--- report_id: {example['report_id']} ---")
            for message in example["messages"]:
                print(f"[{message['role']}] {message['content']}")
            print()

show_examples("data/processed/finetune_extraction_train.jsonl")
show_examples("data/processed/finetune_summarization_train.jsonl")

### Automated validation across every example

A couple of examples looking right doesn't guarantee the other ~3,400 do. This checks all of them for the specific issues we already know to watch for from the EDA: leaked `XXXX` tokens (the redaction cleanup should have caught every occurrence), malformed/incomplete JSON targets for the extraction task, and empty targets for the summarization task (should be zero, since those 6 reports were already excluded when building this file).

In [ ]:
REQUIRED_EXTRACTION_KEYS = {"comparison", "indication", "findings", "impression", "diagnosis"}


def validate_extraction_file(path):
    malformed = 0
    leaked_xxxx = 0
    total = 0
    with open(path) as f:
        for line in f:
            total += 1
            example = json.loads(line)
            roles = [m["role"] for m in example["messages"]]
            assert roles == ["system", "user", "assistant"], f"bad role order: {example['report_id']}"
            for m in example["messages"]:
                if "XXXX" in m["content"]:
                    leaked_xxxx += 1
            try:
                target = json.loads(example["messages"][2]["content"])
                if set(target.keys()) != REQUIRED_EXTRACTION_KEYS:
                    malformed += 1
            except json.JSONDecodeError:
                malformed += 1
    print(f"{path}: {total} examples, {malformed} malformed targets, {leaked_xxxx} messages with leaked XXXX")


def validate_summarization_file(path):
    empty_targets = 0
    leaked_xxxx = 0
    total = 0
    with open(path) as f:
        for line in f:
            total += 1
            example = json.loads(line)
            roles = [m["role"] for m in example["messages"]]
            assert roles == ["system", "user", "assistant"], f"bad role order: {example['report_id']}"
            if not example["messages"][2]["content"].strip():
                empty_targets += 1
            for m in example["messages"]:
                if "XXXX" in m["content"]:
                    leaked_xxxx += 1
    print(f"{path}: {total} examples, {empty_targets} empty targets, {leaked_xxxx} messages with leaked XXXX")


for split in ["train", "val", "test"]:
    validate_extraction_file(f"data/processed/finetune_extraction_{split}.jsonl")
for split in ["train", "val", "test"]:
    validate_summarization_file(f"data/processed/finetune_summarization_{split}.jsonl")

## Next

If both validation functions print 0 malformed/empty/leaked counts across every file, the training data is confirmed clean and we move to the actual QLoRA fine-tuning script: loading **Qwen3-4B** in 4-bit, configuring the LoRA adapter, and training with TRL's `SFTTrainer`, tracked in MLflow. If anything comes back non-zero, bring the exact counts back before we touch training code.